In [15]:
# Load packages

import pandas as pd
import numpy as np

from pathlib import Path
import json
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Keep notebook clean
warnings.filterwarnings("ignore")

print("Imports loaded.")

Imports loaded.


In [29]:
# Reuse / Define Core Helper Functions

def ks_statistic(y_true, y_score):
    temp = pd.DataFrame({
        "y_true": y_true,
        "y_score": y_score
    }).sort_values("y_score", ascending=False)

    temp["good"] = (temp["y_true"] == 0).astype(int)
    temp["bad"] = (temp["y_true"] == 1).astype(int)

    temp["cum_good"] = temp["good"].cumsum() / temp["good"].sum()
    temp["cum_bad"] = temp["bad"].cumsum() / temp["bad"].sum()

    return (temp["cum_bad"] - temp["cum_good"]).abs().max()


def calculate_classification_diagnostics(
    y_train,
    p_train,
    y_val,
    p_val,
    threshold=0.50
):
    rows = []

    for dataset_name, y_true, p_score in [
        ("train", y_train, p_train),
        ("validation", y_val, p_val)
    ]:

        y_pred = (p_score >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        rows.append({
            "dataset": dataset_name,
            "threshold": threshold,
            "auc": roc_auc_score(y_true, p_score),
            "gini": 2 * roc_auc_score(y_true, p_score) - 1,
            "ks": ks_statistic(y_true, p_score),
            "pr_auc": average_precision_score(y_true, p_score),
            "log_loss": log_loss(y_true, p_score),
            "brier_score": brier_score_loss(y_true, p_score),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp
        })

    return pd.DataFrame(rows)

print("Diagnostic helpers loaded.")


def get_next_model_number(model_registry, prefix):
    existing_numbers = []

    for model_num in model_registry.keys():
        if model_num.startswith(prefix):
            number_part = model_num.replace(prefix, "")
            if number_part.isdigit():
                existing_numbers.append(int(number_part))

    return max(existing_numbers) + 1 if existing_numbers else 1


def run_nb_model(
    model_registry,
    model,
    model_number,
    model_name,
    X_train,
    X_val,
    y_train,
    y_val,
    features,
    threshold=0.50,
    analyst_comments="",
    run_permutation=True,
    permutation_scoring="roc_auc",
    permutation_repeats=5,
    display_outputs=True
):

    if model_number in model_registry:
        raise ValueError(f"{model_number} already exists.")

    features = list(features)

    model.fit(X_train[features], y_train)

    p_train = model.predict_proba(X_train[features])[:, 1]
    p_val = model.predict_proba(X_val[features])[:, 1]

    diagnostics_table = calculate_classification_diagnostics(
        y_train=y_train,
        p_train=p_train,
        y_val=y_val,
        p_val=p_val,
        threshold=threshold
    )

    if run_permutation:
        perm_result = permutation_importance(
            model,
            X_val[features],
            y_val,
            scoring=permutation_scoring,
            n_repeats=permutation_repeats,
            random_state=42,
            n_jobs=-1
        )

        permutation_importance_df = pd.DataFrame({
            "variable": features,
            "permutation_importance_mean": perm_result.importances_mean,
            "permutation_importance_std": perm_result.importances_std
        }).sort_values(
            "permutation_importance_mean",
            ascending=False
        ).reset_index(drop=True)
    else:
        permutation_importance_df = pd.DataFrame()

    metadata = {
        "model_number": model_number,
        "model_name": model_name,
        "model_class": type(model).__name__,
        "features": features,
        "num_features": len(features),
        "threshold": threshold,
        "parameters": model.get_params(),
        "analyst_comments": analyst_comments
    }

    model_registry[model_number] = {
        "metadata": metadata,
        "model": model,
        "diagnostics": diagnostics_table,
        "feature_importance": pd.DataFrame(),
        "permutation_importance": permutation_importance_df,
        "p_train": p_train,
        "p_val": p_val
    }

    if display_outputs:
        print("=" * 100)
        print(f"{model_number}: {model_name}")
        print("=" * 100)

        print("\nDIAGNOSTICS:")
        display(diagnostics_table)

        print("\nPERMUTATION IMPORTANCE:")
        display(permutation_importance_df.head(15))

        if analyst_comments:
            print("\nANALYST COMMENTS:")
            print(analyst_comments)

    return model_registry

def save_model_artifact(model_registry, model_id, model_dir, config_dir):
    record = model_registry[model_id]

    model_path = model_dir / f"{model_id}.joblib"
    config_path = config_dir / f"{model_id}_config.json"

    joblib.dump(record["model"], model_path)

    metadata = record.get("metadata", {}).copy()

    config = {
        "model_id": model_id,
        "model_class": str(type(record["model"]).__name__),
        "metadata": metadata,
        "features": metadata.get("features", None),
        "num_features": metadata.get("num_features", None),
    }

    if "diagnostics" in record:
        try:
            config["diagnostics_preview"] = record["diagnostics"].to_dict(orient="records")
        except Exception:
            pass

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4, default=str)

Diagnostic helpers loaded.


In [3]:
# Set directories

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR = OUTPUT_DIR / "saved_models"
CONFIG_DIR = OUTPUT_DIR / "model_configs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)
print("Output Directory:", OUTPUT_DIR)


Project Root: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab
Data Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/data
Output Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs


In [4]:
# Load datasets
df_scaled = pd.read_parquet(DATA_DIR / "df_scaled_model_ready.parquet")

print("Scaled dataset:", df_scaled.shape)

Scaled dataset: (149390, 16)


In [5]:
# Define Target + Candidate Features

target = "SeriousDlqin2yrs"

candidate_features = [
    col for col in df_scaled.columns
    if col != target
]

print("Target:", target)
print("Number of candidate features:", len(candidate_features))
print(candidate_features)

Target: SeriousDlqin2yrs
Number of candidate features: 15
['age', 'age_sq', 'NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfTimes90DaysLate', 'NumberOfOpenCreditLinesAndLoans', 'NumberRealEstateLoansOrLines', 'NumberOfDependents_median', 'NumberOfDependents_missing_flag', 'MonthlyIncome_median', 'MonthlyIncome_missing_flag', 'DebtRatio_log', 'DebtRatio_high_flag', 'RevolvingUtilization_log', 'RevolvingUtilization_high_flag']


In [6]:
# Train / Validation Split
# Use same random_state philosophy as prior logistic notebooks

X = df_scaled[candidate_features].copy()
y = df_scaled[target].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("y_train mean bad rate:", y_train.mean())
print("y_val mean bad rate  :", y_val.mean())

X_train: (104573, 15)
X_val  : (44817, 15)
y_train mean bad rate: 0.06699626098514913
y_val mean bad rate  : 0.06700582368297744


### Note:

#### Why Kitchen Sink for Naive Bayes vs Stepwise Build for Logistic Regression

The feature selection strategy was intentionally different for Logistic Regression and Naive Bayes because the two model families learn from data in fundamentally different ways.

##### Logistic Regression: Incremental / Controlled Variable Addition

For Logistic Regression, variables were often introduced gradually using methods such as IV screening, business logic, and multivariate testing. This approach is appropriate because Logistic Regression estimates explicit coefficients for each feature while assuming a linear relationship in log-odds space.

Adding variables one at a time helps evaluate:

- marginal lift in AUC / KS
- coefficient stability
- multicollinearity effects
- sign consistency
- interpretability
- business justification for inclusion

In short, Logistic Regression benefits from disciplined variable entry because each added feature directly changes the estimated equation.

##### Naive Bayes: Kitchen Sink Baseline First

For Naive Bayes, the starting strategy was different: begin with a broad kitchen sink model using all available candidate features, then prune if necessary.

This is appropriate because Naive Bayes does not estimate one global regression equation. Instead, it models conditional feature distributions within each class and combines them probabilistically under an independence assumption.

As a result:

- weak variables may still add useful likelihood information
- interactions are not explicitly estimated via coefficients
- multicollinearity behaves differently than in Logistic Regression
- some individually modest predictors can still improve class probability estimates
- empirical performance matters more than coefficient-by-coefficient interpretation

Therefore, rather than assuming only top-IV variables matter, it is often better to test a full baseline model first and then compare reduced variants.

##### Practical Summary

- Logistic Regression: selective variable inclusion, one step at a time
- Naive Bayes: broad baseline first, then simplify based on results

In [13]:
# Initialize registry and Gaussian Models

nb_registry = {}

gnb_model = GaussianNB()

In [21]:
# Remove existing GNB models before rerunning models
for key in list(nb_registry.keys()):
    if key.startswith("GNB"):
        del nb_registry[key]

print("Existing GNB models removed.")

Existing GNB models removed.


In [22]:
# GNB001 - Gaussian Naive Bayes Baseline

nb_registry = run_nb_model(
    model_registry=nb_registry,
    model=gnb_model,
    model_number="GNB001",
    model_name="Gaussian Naive Bayes - All Scaled Features",
    X_train=X_train,
    X_val=X_val,
    y_train=y_train,
    y_val=y_val,
    features=candidate_features,
    analyst_comments=(
        "Baseline Gaussian Naive Bayes model using all scaled continuous / engineered features. "
        "This kitchen-sink baseline tests the full conditional-evidence approach before pruning."
    ),
    run_permutation=True,
    permutation_repeats=5,
    display_outputs=True
)

GNB001: Gaussian Naive Bayes - All Scaled Features

DIAGNOSTICS:


,dataset,threshold,auc,gini,ks,pr_auc,log_loss,brier_score,accuracy,precision,recall,f1,true_negative,false_positive,false_negative,true_positive
0,train,0.5,0.775347,0.550694,0.438791,0.234241,0.465944,0.066003,0.931034,0.348974,0.033971,0.061915,97123,444,6768,238
1,validation,0.5,0.773582,0.547163,0.435231,0.236109,0.466864,0.065570,0.931678,0.405145,0.041958,0.076041,41629,185,2877,126



PERMUTATION IMPORTANCE:


,variable,permutation_importance_mean,permutation_importance_std
0,RevolvingUtilization_log,0.118204,0.002164
1,age_sq,0.012419,0.001886
2,NumberOfTime30-59DaysPastDueNotWorse,0.012013,0.000463
3,NumberOfTimes90DaysLate,0.008704,0.000269
4,MonthlyIncome_median,0.007979,0.001076
5,NumberRealEstateLoansOrLines,0.006675,0.000451
6,NumberOfDependents_missing_flag,0.005017,0.002002
7,NumberOfTime60-89DaysPastDueNotWorse,0.003931,0.000256
8,age,0.003847,0.001344
9,NumberOfOpenCreditLinesAndLoans,0.000863,0.000207



ANALYST COMMENTS:
Baseline Gaussian Naive Bayes model using all scaled continuous / engineered features. This kitchen-sink baseline tests the full conditional-evidence approach before pruning.


Gaussian Naive Bayes served as a lightweight probabilistic benchmark. Performance lagged logistic and boosting models, indicating that feature dependence and nonlinear interactions were important drivers of predictive accuracy.

In [23]:
# PCA + GNB set of models

pca_variance_grid = [0.70, 0.80, 0.90, 0.95, 0.99]

start_num = get_next_model_number(nb_registry, "GNB")

for i, variance_cutoff in enumerate(pca_variance_grid, start=start_num):

    model_number = f"GNB{i:03d}"

    gnb_pca_model = Pipeline(
        steps=[
            ("pca", PCA(n_components=variance_cutoff, random_state=42)),
            ("gnb", GaussianNB())
        ]
    )

    nb_registry = run_nb_model(
        model_registry=nb_registry,
        model=gnb_pca_model,
        model_number=model_number,
        model_name=f"Gaussian Naive Bayes - PCA {int(variance_cutoff * 100)}% Variance",
        X_train=X_train,
        X_val=X_val,
        y_train=y_train,
        y_val=y_val,
        features=candidate_features,
        analyst_comments=(
            f"GaussianNB with PCA transformation retaining {variance_cutoff:.0%} variance. "
            "PCA is tested to reduce correlation and redundancy among inputs, "
            "which may better align with Naive Bayes independence assumptions."
        ),
        run_permutation=True,
        permutation_repeats=5,
        display_outputs=False
    )

print("PCA + GaussianNB grid complete.")

PCA + GaussianNB grid complete.


In [24]:
# Naive Bayes Summary Table

nb_summary_rows = []

for model_num, model_data in nb_registry.items():

    meta = model_data["metadata"]
    diag = model_data["diagnostics"]

    train_row = diag.loc[diag["dataset"] == "train"].iloc[0]
    val_row = diag.loc[diag["dataset"] == "validation"].iloc[0]

    params = meta.get("parameters", {})

    # Pull PCA details if model is a pipeline
    pca_components = None
    pca_variance_sum = None

    if hasattr(model_data["model"], "named_steps") and "pca" in model_data["model"].named_steps:
        pca_step = model_data["model"].named_steps["pca"]
        pca_components = pca_step.n_components_
        pca_variance_sum = pca_step.explained_variance_ratio_.sum()

    nb_summary_rows.append({
        "model_number": model_num,
        "model_name": meta["model_name"],
        "model_class": meta["model_class"],
        "pca_components": pca_components,
        "pca_variance_sum": pca_variance_sum,
        "train_auc": train_row["auc"],
        "val_auc": val_row["auc"],
        "auc_gap": train_row["auc"] - val_row["auc"],
        "train_ks": train_row["ks"],
        "val_ks": val_row["ks"],
        "ks_gap": train_row["ks"] - val_row["ks"],
        "val_pr_auc": val_row["pr_auc"],
        "val_brier": val_row["brier_score"],
        "val_log_loss": val_row["log_loss"]
    })

nb_summary_df = (
    pd.DataFrame(nb_summary_rows)
    .sort_values(["val_auc", "val_ks"], ascending=False)
    .reset_index(drop=True)
)

display(nb_summary_df)

,model_number,model_name,model_class,pca_components,pca_variance_sum,train_auc,val_auc,auc_gap,train_ks,val_ks,ks_gap,val_pr_auc,val_brier,val_log_loss
0,GNB001,Gaussian Naive Bayes - All Scaled Features,GaussianNB,NaN,NaN,0.775347,0.773582,0.001765,0.438791,0.435231,0.003560,0.236109,0.065570,0.466864
1,GNB005,Gaussian Naive Bayes - PCA 95% Variance,Pipeline,9.0,0.951169,0.765282,0.762130,0.003152,0.438597,0.430280,0.008316,0.218701,0.062068,0.342247
2,GNB006,Gaussian Naive Bayes - PCA 99% Variance,Pipeline,11.0,0.994924,0.764060,0.760525,0.003535,0.437993,0.431662,0.006331,0.219160,0.061806,0.342975
3,GNB003,Gaussian Naive Bayes - PCA 80% Variance,Pipeline,7.0,0.847313,0.728547,0.724516,0.004031,0.363903,0.363888,0.000015,0.201522,0.064073,0.358056
4,GNB004,Gaussian Naive Bayes - PCA 90% Variance,Pipeline,8.0,0.902124,0.718108,0.714726,0.003382,0.341778,0.344415,-0.002637,0.177187,0.064250,0.359863
5,GNB002,Gaussian Naive Bayes - PCA 70% Variance,Pipeline,5.0,0.720088,0.681965,0.677698,0.004268,0.272900,0.279107,-0.006207,0.182949,0.062332,0.281147


We tested PCA to reduce feature dependence and better align with Naive Bayes assumptions. However, performance consistently declined, indicating that the original variables contained more discriminative structure than unsupervised latent components.

In [26]:
# Variable Ranking from Original GNB Baseline

ranked_vars = (
    nb_registry["GNB001"]["permutation_importance"]   # rename to your raw model id
    ["variable"]
    .tolist()
)

In [27]:
# Pruned GaussianNB Grid

top_n_grid = [5, 8, 10, 12]

start_num = get_next_model_number(nb_registry, "GNB")

for i, top_n in enumerate(top_n_grid, start=start_num):

    model_number = f"GNB{i:03d}"

    selected_features = ranked_vars[:top_n]

    gnb_model = GaussianNB()

    nb_registry = run_nb_model(
        model_registry=nb_registry,
        model=gnb_model,
        model_number=model_number,
        model_name=f"Gaussian Naive Bayes - Top {top_n} Variables",
        X_train=X_train,
        X_val=X_val,
        y_train=y_train,
        y_val=y_val,
        features=selected_features,
        analyst_comments=(
            f"Pruned GaussianNB using top {top_n} variables ranked by "
            "baseline permutation importance."
        ),
        run_permutation=True,
        permutation_repeats=5,
        display_outputs=False
    )

print("Pruned GaussianNB grid complete.")

Pruned GaussianNB grid complete.


In [28]:
# Final Naive Bayes Summary Table

nb_summary_rows = []

for model_num, model_data in nb_registry.items():

    meta = model_data["metadata"]
    diag = model_data["diagnostics"]

    train_row = diag.loc[diag["dataset"] == "train"].iloc[0]
    val_row   = diag.loc[diag["dataset"] == "validation"].iloc[0]

    # Defaults
    pca_components = None
    pca_variance_sum = None
    num_features = meta.get("num_features", None)

    # Detect PCA pipeline
    model_obj = model_data["model"]

    if hasattr(model_obj, "named_steps") and "pca" in model_obj.named_steps:
        pca_step = model_obj.named_steps["pca"]
        pca_components = pca_step.n_components_
        pca_variance_sum = round(
            pca_step.explained_variance_ratio_.sum(), 6
        )

    nb_summary_rows.append({
        "model_number": model_num,
        "model_name": meta["model_name"],
        "model_class": meta["model_class"],
        "num_features": num_features,
        "pca_components": pca_components,
        "pca_variance_sum": pca_variance_sum,
        "train_auc": train_row["auc"],
        "val_auc": val_row["auc"],
        "auc_gap": train_row["auc"] - val_row["auc"],
        "train_ks": train_row["ks"],
        "val_ks": val_row["ks"],
        "ks_gap": train_row["ks"] - val_row["ks"],
        "val_pr_auc": val_row["pr_auc"],
        "val_brier": val_row["brier_score"],
        "val_log_loss": val_row["log_loss"]
    })

nb_summary_df = (
    pd.DataFrame(nb_summary_rows)
    .sort_values(
        by=["val_auc", "val_ks"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(nb_summary_df)

,model_number,model_name,model_class,num_features,pca_components,pca_variance_sum,train_auc,val_auc,auc_gap,train_ks,val_ks,ks_gap,val_pr_auc,val_brier,val_log_loss
0,GNB007,Gaussian Naive Bayes - Top 5 Variables,GaussianNB,5,NaN,NaN,0.797053,0.798680,-0.001628,0.472268,0.473626,-0.001358,0.296942,0.060211,0.289474
1,GNB008,Gaussian Naive Bayes - Top 8 Variables,GaussianNB,8,NaN,NaN,0.795913,0.796611,-0.000698,0.474062,0.473531,0.000530,0.292547,0.064061,0.376659
2,GNB009,Gaussian Naive Bayes - Top 10 Variables,GaussianNB,10,NaN,NaN,0.786138,0.786717,-0.000579,0.463796,0.464021,-0.000224,0.277121,0.063384,0.370385
3,GNB010,Gaussian Naive Bayes - Top 12 Variables,GaussianNB,12,NaN,NaN,0.782455,0.783828,-0.001373,0.454499,0.455182,-0.000684,0.252282,0.065775,0.466010
4,GNB001,Gaussian Naive Bayes - All Scaled Features,GaussianNB,15,NaN,NaN,0.775347,0.773582,0.001765,0.438791,0.435231,0.003560,0.236109,0.065570,0.466864
5,GNB005,Gaussian Naive Bayes - PCA 95% Variance,Pipeline,15,9.0,0.951169,0.765282,0.762130,0.003152,0.438597,0.430280,0.008316,0.218701,0.062068,0.342247
6,GNB006,Gaussian Naive Bayes - PCA 99% Variance,Pipeline,15,11.0,0.994924,0.764060,0.760525,0.003535,0.437993,0.431662,0.006331,0.219160,0.061806,0.342975
7,GNB003,Gaussian Naive Bayes - PCA 80% Variance,Pipeline,15,7.0,0.847313,0.728547,0.724516,0.004031,0.363903,0.363888,0.000015,0.201522,0.064073,0.358056
8,GNB004,Gaussian Naive Bayes - PCA 90% Variance,Pipeline,15,8.0,0.902124,0.718108,0.714726,0.003382,0.341778,0.344415,-0.002637,0.177187,0.064250,0.359863
9,GNB002,Gaussian Naive Bayes - PCA 70% Variance,Pipeline,15,5.0,0.720088,0.681965,0.677698,0.004268,0.272900,0.279107,-0.006207,0.182949,0.062332,0.281147


In [30]:
# Save Naive Bayes Champion Model Artifact

save_model_artifact(
    model_registry=nb_registry,
    model_id="GNB007",
    model_dir=MODEL_DIR,
    config_dir=CONFIG_DIR
)

In [31]:
# Export Naive Bayes Champion Summary

nb_champion_id = "GNB007"

nb_champion_path = OUTPUT_DIR / "04c1_nb_gaussian_champion_summary.xlsx"

with pd.ExcelWriter(nb_champion_path, engine="openpyxl") as writer:
    nb_summary_df.to_excel(
        writer,
        sheet_name="NB_Model_Comparison",
        index=False
    )

    nb_registry[nb_champion_id]["diagnostics"].to_excel(
        writer,
        sheet_name="GNB007_Diagnostics",
        index=False
    )

    nb_registry[nb_champion_id]["permutation_importance"].to_excel(
        writer,
        sheet_name="GNB007_Permutation",
        index=False
    )

print("Saved:", nb_champion_path)

Saved: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/04c1_nb_gaussian_champion_summary.xlsx


## Gaussian Naive Bayes Summary

Gaussian Naive Bayes was tested as a lightweight probabilistic benchmark and as a contrasting model family to Logistic Regression and tree-based methods. Unlike regression models that estimate a single global equation, Naive Bayes combines class-conditional evidence across variables under an independence assumption. This made it a useful benchmark to test whether a simpler probability-based framework could perform competitively on the credit-risk problem.

The full-feature baseline model underperformed relative to Logistic Regression and boosting models, which was not unexpected. Credit variables such as utilization, delinquency counts, debt burden, and credit line exposure are naturally correlated, meaning the conditional independence assumption is violated. In practice, this can cause Naive Bayes to overweight redundant information or dilute the strongest signals.

PCA-based variants were then tested to reduce correlation and better align the data with Naive Bayes assumptions. However, performance deteriorated across PCA models. This suggests that while PCA removed redundancy, it also blended original variables into unsupervised components that preserved variance rather than the strongest default-risk separation signal.

The strongest Naive Bayes result came from supervised feature pruning. Using variables ranked by baseline permutation importance, the Top 5 variable model (GNB007) materially improved validation AUC versus the kitchen-sink baseline. This indicates that Naive Bayes can remain effective when restricted to a compact set of strong, relatively distinct predictors, while excessive noisy or overlapping variables reduce performance.

The final Gaussian Naive Bayes champion was **GNB007**. Although it did not outperform the leading Logistic or boosting models, it served as a valuable benchmark and demonstrated an important modeling lesson: simpler algorithms can become competitive when their assumptions are respected and feature selection is disciplined.

**Final Gaussian NB champion:** GNB007  

**Best PCA variant:** GNB005 (95% variance)  

**Key takeaway:** supervised pruning outperformed unsupervised dimensionality reduction.